In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset

print("Libraries imported successfully!")
print("GPU available:", torch.cuda.is_available())

Libraries imported successfully!
GPU available: False


In [11]:
df=pd.read_csv('/content/Reviews.csv')

In [12]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [13]:
df = df.dropna(subset=["Score", "Text"]).copy()
df = df.drop_duplicates(subset=["Text", "Score"])

df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df = df.dropna(subset=["Score"])
df = df[df["Score"].isin([1, 2, 3, 4, 5])]

def sentiment(score):
    if score <= 2:
        return 0       # Negative
    elif score == 3:
        return 1       # Neutral
    return 2           # Positive

df["label"] = df["Score"].apply(sentiment)
df["Text"] = df["Text"].astype(str)

# Balanced sample: up to 3,000 reviews per class
df = (
    df.groupby("label", group_keys=False)
      .sample(n=min(3000, df["label"].value_counts().min()),
              random_state=42)
      .reset_index(drop=True)
)

print(df["label"].value_counts())
print("Dataset shape:", df.shape)

label
0    3000
1    3000
2    3000
Name: count, dtype: int64
Dataset shape: (9000, 11)


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    df["Text"], df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 7200
Testing samples: 1800


In [15]:
from tensorflow.keras.layers import TextVectorization, Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

vectorizer = TextVectorization(
    max_tokens=20000,
    output_sequence_length=150
)

vectorizer.adapt(
    tf.data.Dataset.from_tensor_slices(X_train.to_numpy()).batch(64)
)

X_train_seq = vectorizer(np.array(X_train))
X_test_seq = vectorizer(np.array(X_test))

lstm_model = Sequential([
    Embedding(20000, 64),
    LSTM(64),
    Dropout(0.3),
    Dense(3, activation="softmax")
])

lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
lstm_model.fit(
    X_train_seq,
    np.array(y_train),
    epochs=3,
    batch_size=64,
    validation_split=0.1
)

lstm_pred = np.argmax(
    lstm_model.predict(X_test_seq, verbose=0), axis=1
)

print("LSTM Accuracy:", accuracy_score(y_test, lstm_pred))
print(classification_report(
    y_test, lstm_pred,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

Epoch 1/3
102/102 ━━━━━━━━━━━━━━━━━━━━ 34s 289ms/step - accuracy: 0.3366 - loss: 1.0988 - val_accuracy: 0.3875 - val_loss: 1.0940
Epoch 2/3
102/102 ━━━━━━━━━━━━━━━━━━━━ 17s 165ms/step - accuracy: 0.3525 - loss: 1.0948 - val_accuracy: 0.3458 - val_loss: 1.0961
Epoch 3/3
102/102 ━━━━━━━━━━━━━━━━━━━━ 14s 136ms/step - accuracy: 0.3610 - loss: 1.0854 - val_accuracy: 0.3597 - val_loss: 1.1056
LSTM Accuracy: 0.365
              precision    recall  f1-score   support

    Negative       0.34      0.78      0.48       600
     Neutral       0.43      0.31      0.36       600
    Positive       0.00      0.00      0.00       600

    accuracy                           0.36      1800
   macro avg       0.26      0.36      0.28      1800
weighted avg       0.26      0.36      0.28      1800



In [17]:
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_data = Dataset.from_dict({
    "text": X_train.tolist(),
    "labels": y_train.tolist()
})

test_data = Dataset.from_dict({
    "text": X_test.tolist(),
    "labels": y_test.tolist()
})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data = train_data.remove_columns(["text"])
test_data = test_data.remove_columns(["text"])

train_data.set_format("torch")
test_data.set_format("torch")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label={0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"},
    label2id={"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
)

training_args = TrainingArguments(
    output_dir="./sentiment_results",
    num_train_epochs=1,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer
)

trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            